# Open-Loop Reconfiguration Test

Run a scripted sequence of R/T/N burns through `RecfgEnv` and plot the ROE evolution to sanity-check axis mapping and reward shaping.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from leo_gym.gyms.recfg_gym import RecfgEnv, RecfgEnvConfig
from leo_gym.orbit.dynamics.dynamics import DynamicsConfig
from leo_gym.satellite.satellite_roe import SatelliteROEConfig
from leo_gym.utils.utils import gen_rv0, seed_all

In [ ]:
def make_satellite_cfg(seed: int | None = None) -> SatelliteROEConfig:
    dyn_kwargs = dict(
        flag_rtn_thrust=True,
        flag_mass_loss=False,
        flag_pert_moon=False,
        flag_pert_sun=False,
        flag_pert_srp=False,
        flag_pert_drag=False,
        flag_pert_irr_grav=False,
        eph_time_0=6.338304141847866e8,
        m=150.0,
        f_max=0.018,
        Isp=860.0,
        Ad=1.3,
        Cd=2.2,
        Cr=1.3,
        As=1.3,
        mf=136.0,
    )
    if seed is not None:
        seed_all(seed)
    return SatelliteROEConfig(
        dt=60.0,
        days=1.0,
        ideal_traj_params=DynamicsConfig(**dyn_kwargs),
        pert_traj_params=DynamicsConfig(**dyn_kwargs),
        rv0=gen_rv0(sma=7000e3),
    )

def make_env_cfg(sat_cfg: SatelliteROEConfig) -> RecfgEnvConfig:
    return RecfgEnvConfig(
        low_action=[0, 1],
        high_action=[0, 800],  # allow long burns (minutes)
        satellite_config=sat_cfg,
        satellite_observation_feature_size=6,
        continuous_actions_size=2,
        discrete_actions_size=7,
        f_max=sat_cfg.pert_traj_params.f_max,
        max_time_index=2000,  # steps; with dt=60s -> ~33 hours horizon
        target_roe=None,
        target_tolerance=25.0,
        reward_distance_scale=1e4,
        success_reward=10.0,
        fuel_penalty_weight=1e-2,
    )

def run_scripted_plan(env: RecfgEnv, plan, reset_seed: int | None = None):
    """Execute a list of (axis_id, delay_steps, duration_steps) actions."""
    obs_history, raw_roe_history, reward_history, info_history, n_history = [], [], [], [], []
    thrust_history = None
    if reset_seed is not None:
        obs, info = env.reset(seed=reset_seed)
    else:
        obs, info = env.reset()
    obs_history.append(obs)
    raw_roe_history.append(np.array(env.satellite.roe[-1]))
    info_history.append(info)
    n_history.append(info.get("n", getattr(env.satellite, "discrete_time_index_simulation", 0)))

    for axis_id, delay, dur in plan:
        action = {"discrete": axis_id, "continuous": np.array([delay, dur], dtype=np.float64)}
        obs, reward, terminated, truncated, info = env.step(action)
        obs_history.append(obs)
        raw_roe_history.append(np.array(env.satellite.roe[-1]))
        reward_history.append(reward)
        info_history.append(info)
        n_history.append(info.get("n", getattr(env.satellite, "discrete_time_index_simulation", 0)))
        if terminated or truncated:
            break

    n_end = getattr(env.satellite, "discrete_time_index_simulation", len(n_history))
    thrust_arr = getattr(env.satellite, "manplan_control_inputs", None)
    if thrust_arr is None or len(np.array(thrust_arr)) == 0:
        thrust_arr = getattr(env.satellite, "thrust_rtn", None)
    if thrust_arr is not None:
        thrust_history = np.array(thrust_arr)[:n_end]

    return {
        "obs_history": np.array(obs_history),
        "raw_roe_history": np.array(raw_roe_history),
        "reward_history": np.array(reward_history),
        "info_history": info_history,
        "n_history": np.array(n_history),
        "thrust_history": thrust_history,
    }

def plot_roe(history: np.ndarray, n_history: np.ndarray, dt_seconds: float, title: str) -> None:
    t = np.array(n_history) * dt_seconds / 60.0  # minutes from sim index
    labels = ["a·δa", "a·δλ", "a·δe_x", "a·δe_y", "a·δi_x", "a·δi_y"]
    inc_norm = None
    fig = make_subplots(rows=3, cols=2, shared_xaxes=True, subplot_titles=labels)
    if history.shape[1] >= 6:
        inc_norm = np.sqrt(history[:, 4]**2 + history[:, 5]**2)
    for i, label in enumerate(labels):
        if i >= history.shape[1]:
            break
        row = i // 2 + 1
        col = i % 2 + 1
        fig.add_trace(go.Scatter(x=t, y=history[:, i], mode="lines", name=label), row=row, col=col)
    fig.update_layout(height=800, width=900, title_text=title, showlegend=False)
    fig.update_xaxes(title_text="Time [min]", row=3, col=1)
    fig.update_xaxes(title_text="Time [min]", row=3, col=2)
    fig.show()
    if inc_norm is not None:
        fig_norm = go.Figure()
        fig_norm.add_trace(go.Scatter(x=t, y=inc_norm, mode="lines", name="||a·δi||"))
        fig_norm.update_layout(title="Inclination vector norm", xaxis_title="Time [min]", yaxis_title="||a·δi||")
        fig_norm.show()

def plot_thrust(thrust_history, dt_seconds: float) -> None:
    if thrust_history is None:
        return
    thr = np.array(thrust_history)
    t_thr = np.arange(thr.shape[0]) * dt_seconds / 60.0
    max_mag = float(np.max(np.abs(thr))) if thr.size else None
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=["R", "T", "N"])
    for i, label in enumerate(["R", "T", "N"]):
        if i >= thr.shape[1]:
            break
        fig.add_trace(go.Scatter(x=t_thr, y=thr[:, i], mode="lines", name=label), row=i+1, col=1)
        if max_mag:
            fig.update_yaxes(range=[-max_mag, max_mag], row=i+1, col=1)
    fig.update_layout(height=700, title_text="Thrust (RTN) vs time")
    fig.update_xaxes(title_text="Time [min]", row=3, col=1)
    fig.show()


In [ ]:
# Set SEED to an int for reproducible runs, or None for random each time.
SEED = 42

# axis_id map from RecfgEnv: 0:+R,1:-R,2:+T,3:-T,4:+N,5:-N,6:coast
sat_cfg = make_satellite_cfg(seed=SEED)
env_cfg = make_env_cfg(sat_cfg)
env_seed = SEED if SEED is not None else np.random.randint(0, 2**32 - 1)
env = RecfgEnv(env_cfg, seed=env_seed)

# Example long +N burn
scripted = [
    (4, 0, 500),
]

result = run_scripted_plan(env, scripted, reset_seed=env_seed)
print(f"Applied steps: {len(result.get('thrust_history', []))}")
print(f"f_max: {env_cfg.f_max}, dt: {sat_cfg.dt}")
print("dyn f_max:", env.satellite.dynamics1.cfg.f_max)
print("mass:", env.satellite.rvm_eci[-1][6])
thr = result["thrust_history"]
dv = np.trapz(thr[:,2] / env.satellite.rvm_eci[-1][6], dx=sat_cfg.dt)
print("Δv from thrust history (normal axis):", dv, "m/s")

if result.get("thrust_history") is not None:
    print("Last 3 thrust vectors:", result["thrust_history"][-3:])
plot_roe(result["raw_roe_history"], result["n_history"], dt_seconds=sat_cfg.dt, title="ROE evolution (raw)")
plot_roe(result["obs_history"], result["n_history"], dt_seconds=sat_cfg.dt, title="ROE evolution (obs)")
plot_thrust(result.get("thrust_history"), dt_seconds=sat_cfg.dt)

